# Análise e tratamento inicial do PeerRead

O ZIP referenciado pelo loader da página Hugging Face foi baixado em raw/PeerRead-master.zip e extraído em raw/PeerRead-master/. O SHA-256 confere com o checksum dos metadados. Este notebook analisa parsed_pdfs e reviews, mantém conferência e partição como proveniência e trata cópias em memória; os originais não são alterados.

Requer Python, pandas e IPython no kernel.

In [2]:
from pathlib import Path
from copy import deepcopy
import json
import pandas as pd
from IPython.display import display, Markdown

cwd = Path.cwd()
DATASET_DIR = cwd / 'dataset_project' if (cwd / 'dataset_project').is_dir() else cwd
DATA_DIR = DATASET_DIR / 'raw' / 'PeerRead-master' / 'data'
INFO_PATH = DATASET_DIR / 'dataset_infos.json'
print('Diretório do projeto:', DATASET_DIR.resolve())
print('Dados extraídos:', DATA_DIR.exists())

Diretório do projeto: /home/iara-batista/Documentos/Faculdade/taes/metis_system/dataset_project
Dados extraídos: True


## 1. Inventário e carregamento

In [3]:
json_files = sorted(p for config in ('parsed_pdfs', 'reviews')
                    for p in DATA_DIR.glob(f'*/*/{config}/*.json')) if DATA_DIR.exists() else []
print(f'JSONs selecionados: {len(json_files):,}')
records, read_errors = [], []
for path in json_files:
    try:
        payload = json.loads(path.read_text(encoding='utf-8'))
        batch = payload if isinstance(payload, list) else [payload]
        for position, row in enumerate(batch):
            if isinstance(row, dict):
                records.append({'_source_file': str(path.relative_to(DATASET_DIR)),
                                '_record_in_file': position, '_config': path.parent.name,
                                '_split': path.parent.parent.name,
                                '_conference': path.parent.parent.parent.name, **row})
            else:
                read_errors.append({'arquivo': str(path), 'erro': f'registro {position} não é objeto'})
    except Exception as exc:
        read_errors.append({'arquivo': str(path), 'erro': repr(exc)})
raw_df = pd.json_normalize(records, sep='.') if records else pd.DataFrame()
print(f'Registros lidos: {len(raw_df):,}; arquivos/linhas inválidos: {len(read_errors)}')
if read_errors:
    display(pd.DataFrame(read_errors).head(30))
if records:
    display(raw_df.groupby(['_config', '_split']).size().rename('registros').to_frame())
    display(raw_df.groupby(['_config', '_conference']).size().rename('registros').to_frame())

JSONs selecionados: 24,728
Registros lidos: 24,728; arquivos/linhas inválidos: 0


registros
_config     _split           
parsed_pdfs dev           637
            test          637
            train       11090
reviews     dev           637
            test          637
            train       11090

registros
_config     _conference                     
parsed_pdfs acl_2017                     137
            arxiv.cs.ai_2007-2017       4092
            arxiv.cs.cl_2007-2017       2638
            arxiv.cs.lg_2007-2017       5048
            conll_2016                    22
            iclr_2017                    427
reviews     acl_2017                     137
            arxiv.cs.ai_2007-2017       4092
            arxiv.cs.cl_2007-2017       2638
            arxiv.cs.lg_2007-2017       5048
            conll_2016                    22
            iclr_2017                    427

## 2. IDs e vazamento entre partições

Os IDs locais podem repetir entre conferências. A chave usada abaixo é conferência + ID; juntar somente pelo ID pode associar artigos diferentes.

In [4]:
id_rows = []
for row in records:
    if row['_config'] == 'parsed_pdfs':
        paper_id = str(row.get('name') or Path(row['_source_file']).name).removesuffix('.pdf')
    else:
        paper_id = row.get('id')
    if paper_id is not None:
        id_rows.append({'config': row['_config'], 'conference': row['_conference'],
                        'paper_id': str(paper_id), 'split': row['_split']})
id_df = pd.DataFrame(id_rows)
if not id_df.empty:
    scoped = id_df.groupby(['config', 'conference', 'paper_id']).agg(
        records=('split', 'size'), split_count=('split', 'nunique'))
    collisions = id_df.groupby(['config', 'paper_id']).conference.nunique()
    unique_counts = id_df.drop_duplicates(['config', 'conference', 'paper_id']).groupby('config').size().to_dict()
    collision_counts = collisions.gt(1).groupby(level='config').sum().to_dict()
    split_membership = id_df.groupby(['config', 'conference', 'paper_id']).split.agg(lambda s: tuple(sorted(set(s))))
    print('IDs únicos com chave conferência + ID:', unique_counts)
    print('IDs repetidos na mesma conferência:', int((scoped.records > 1).sum()))
    print('IDs locais em colisão entre conferências, por configuração:', collision_counts)
    print('Artigos repetidos entre partições na mesma conferência:', int((split_membership.map(len) > 1).sum()))
    display(scoped[scoped.records > 1].head(20))
    display(split_membership[split_membership.map(len) > 1].head(20).to_frame('partições'))

IDs únicos com chave conferência + ID: {'parsed_pdfs': 12364, 'reviews': 12364}
IDs repetidos na mesma conferência: 0
IDs locais em colisão entre conferências, por configuração: {'parsed_pdfs': 71, 'reviews': 71}
Artigos repetidos entre partições na mesma conferência: 0


,,,records,split_count
config,conference,paper_id,,


,,,partições
config,conference,paper_id,


## 3. Qualidade dos artigos parseados

In [5]:
paper_rows = [r for r in records if r['_config'] == 'parsed_pdfs']
structure, quality = [], []
for row in paper_rows:
    meta = row.get('metadata') if isinstance(row.get('metadata'), dict) else {}
    structure.append({'ano': meta.get('year'), 'seções': len(meta.get('sections') or []),
                      'referências': len(meta.get('references') or []),
                      'menções': len(meta.get('referenceMentions') or []),
                      'abstract_chars': len(meta.get('abstractText') or '') if isinstance(meta.get('abstractText'), str) else None})
    fields = {'título': meta.get('title'), 'autores': meta.get('authors'),
              'referências': meta.get('references'), 'menções': meta.get('referenceMentions'),
              'abstract': meta.get('abstractText'), 'seções': meta.get('sections'), 'ano': meta.get('year')}
    for field, value in fields.items():
        quality.append({'campo': field, 'ausente': value is None or value == '' or value == []})
if structure:
    display(pd.DataFrame(structure).describe().T)
    quality_df = pd.DataFrame(quality).groupby('campo').ausente.agg(['sum', 'mean'])
    quality_df.columns = ['registros ausentes', 'ausentes (%)']
    quality_df['ausentes (%)'] = (quality_df['ausentes (%)'] * 100).round(2)
    display(quality_df.sort_values('registros ausentes', ascending=False))

,count,mean,std,min,25%,50%,75%,max
ano,12364.0,2009.739000,104.107709,0.0,2014.0,2016.0,2017.0,2553.0
seções,12364.0,14.893966,11.343368,0.0,9.0,13.0,18.0,253.0
referências,12364.0,24.479295,19.645824,0.0,13.0,22.0,32.0,559.0
menções,12364.0,34.957053,97.548579,0.0,12.0,28.0,47.0,10169.0
abstract_chars,12209.0,1114.841183,1260.470751,1.0,746.0,968.0,1234.0,68711.0


,registros ausentes,ausentes (%)
campo,,
menções,1589,12.85
título,1502,12.15
autores,862,6.97
referências,853,6.90
seções,177,1.43
abstract,155,1.25
ano,0,0.00


## 4. Tratamento conservador e perfil dos reviews

In [6]:
MISSING_MARKERS = {'', 'na', 'n/a', 'none', 'null', 'nan', 'unknown'}
def clean_value(value):
    if isinstance(value, dict):
        return {k: clean_value(v) for k, v in value.items()}
    if isinstance(value, list):
        return [clean_value(v) for v in value]
    if isinstance(value, str):
        value = value.strip()
        return None if value.casefold() in MISSING_MARKERS else value
    return value
clean_records = [clean_value(deepcopy(r)) for r in records]
clean_df = pd.json_normalize(clean_records, sep='.') if clean_records else pd.DataFrame()
print('Cópia tratada em memória; dados brutos preservados.')

score_names = ['originality', 'soundness_correctness', 'substance', 'clarity',
               'meaningful_comparison', 'impact', 'replicability', 'appropriateness',
               'recommendation', 'reviewer_confidence']
review_rows, paper_review_rows = [], []
for row in clean_records:
    if row['_config'] != 'reviews':
        continue
    reviews = row.get('reviews') or []
    if isinstance(reviews, dict):
        reviews = [reviews]
    article_id = row.get('id') or row.get('_source_file')
    paper_review_rows.append({'article_id': article_id, 'conference': row['_conference'],
                              'split': row['_split'], 'accepted': row.get('accepted'),
                              'review_count': len(reviews)})
    for n, review in enumerate(reviews):
        if isinstance(review, dict):
            normalized = {str(k).casefold(): v for k, v in review.items()}
            review_rows.append({'article_id': article_id, 'conference': row['_conference'],
                                'split': row['_split'], 'review_index': n, **normalized})
review_df = pd.DataFrame(review_rows)
paper_reviews_df = pd.DataFrame(paper_review_rows)
print(f'Artigos em reviews: {len(paper_reviews_df):,}')
print(f'Artigos com avaliações: {(paper_reviews_df.review_count > 0).sum():,}; sem avaliação: {(paper_reviews_df.review_count == 0).sum():,}')
print(f'Reviews individuais expandidos: {len(review_df):,}')
if not review_df.empty:
    display(paper_reviews_df.groupby('split').agg(
        artigos=('article_id', 'count'), aceitos=('accepted', lambda s: (s == True).sum()),
        rejeitados=('accepted', lambda s: (s == False).sum()),
        aceite_ausente=('accepted', lambda s: s.isna().sum()),
        artigos_com_review=('review_count', lambda s: (s > 0).sum())))
    for col in [c for c in score_names if c in review_df.columns]:
        print('\n', col)
        display(review_df[col].value_counts(dropna=False).rename_axis('valor').to_frame('contagem'))

Cópia tratada em memória; dados brutos preservados.
Artigos em reviews: 12,364
Artigos com avaliações: 586; sem avaliação: 11,778
Reviews individuais expandidos: 7,584


,artigos,aceitos,rejeitados,aceite_ausente,artigos_com_review
split,,,,,
dev,637,156,473,8,48
test,637,153,475,9,47
train,11090,2754,8194,142,491



 originality


,contagem
valor,
NaN,6754
4,186
3,139
2,136
3,98
5,90
5,80
4,76
1,16



 soundness_correctness


,contagem
valor,
NaN,6882
5,160
5,131
3,110
3,108
4,84
4,70
2,36
2,3


... (12 outputs left out)


 recommendation


,contagem
valor,
NaN,4628
6,600
7,572
5,464
4,454
8,246
3,214
4,156
3,79



 reviewer_confidence


,contagem
valor,
NaN,4652
4,1410
3,678
5,428
4,180
2,88
3,66
5,52
2,15


## 5. Observações sobre o loader legado

- Os arquivos de reviews guardam as avaliações na chave superior reviews, mas o método _parse_reviews procura metadata.reviews e pode descartá-las.
- Os aspectos aparecem em maiúsculas (ORIGINALITY, SOUNDNESS_CORRECTNESS), enquanto o loader procura nomes minúsculos; isso pode produzir campos vazios.
- Campos declarados bool/int32 são convertidos em strings em partes do loader; ausências podem virar string vazia ou string None.
- IDs numéricos locais colidem entre conferências. Use conferência + ID para junções.
- As notas aparecem como strings categóricas: vários aspectos usam 1–5, recommendation chega a 10. Não combine escalas sem normalizar por conferência.

## 6. Resultado da análise inicial

- Há 12.364 registros em cada configuração (parsed_pdfs e reviews): 11.090 treino, 637 dev e 637 teste. As pastas usam dev; os metadados do Hugging Face usam validation.
- Há 12.364 chaves únicas por configuração usando conferência + ID. Não há repetição na mesma conferência nem artigos em partições diferentes dentro da conferência. O ID isolado colide entre 71 registros de conferências diferentes; o ID 489 em ACL e ICLR, por exemplo, tem títulos distintos.
- Em parsed_pdfs, a mediana é 13 seções, 22 referências e 28 menções de referência. Máximos: 253 seções, 559 referências e 10.169 menções; revisar esses extremos antes de filtros.
- Campos vazios em parsed_pdfs: título 1.502 (12,1%), autores 862 (7,0%), referências 853 (6,9%), menções 1.589 (12,9%), abstract 155 (1,3%), seções 177 (1,4%) e ano 33 (0,3%). Ausência não mede qualidade semântica.
- 586 artigos têm pelo menos um review, somando 7.584 reviews; 11.778 registros não têm reviews anotados.
- A decisão de aceite está ausente em 159 linhas; há 9.142 False e 3.063 True.
- As notas são rótulos numéricos armazenados como texto, em geral na faixa 1–5; recommendation usa valores até 10. Há muitos campos de nota ausentes.

O tratamento remove espaços periféricos e converte marcadores de ausência em nulos numa cópia em memória. Não remove registros nem recodifica as escalas. Para calibração, considerar os aspectos ausentes e diferenças entre conferências.